# RehabLLM — Free GPU Training (Google Colab + Kaggle)

This single notebook trains the **17.4M-parameter RehabLLM** on either Google Colab or Kaggle free GPU runtimes.

**Pipeline:** environment detection → GPU check → clone/install → corpus build/restore → tokenizer → train/val/test tokenisation → resumable GPU training → evaluation → artefact packaging.

> Research use only. RehabLLM is not a clinically validated model and must not be used for diagnosis, treatment selection, or patient-specific medical decisions.


## 1. Settings

The defaults run the substantial free-GPU experiment. You normally do not need to change anything here.


In [ ]:
REPO_URL = "https://github.com/ZenKOH/RehabLLM.git"
REPO_BRANCH = "main"
TARGET_DOCS = 12_000
CONFIG_PATH = "configs/free_gpu.yaml"
PERSIST_FOLDER = "RehabLLM-FreeGPU"
FORCE_REBUILD_DATA = False
EVAL_PROMPT = "Explain robot-assisted rehabilitation after stroke, including important evidence limitations."

print({
    "repo": REPO_URL,
    "branch": REPO_BRANCH,
    "target_docs": TARGET_DOCS,
    "config": CONFIG_PATH,
})


## 2. Detect Colab/Kaggle and choose persistent storage

On **Colab**, checkpoints and prepared data are stored in Google Drive so they survive runtime restarts. On **Kaggle**, they are stored under `/kaggle/working`; save a notebook version to retain outputs between sessions.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle").exists() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content")
    PERSIST_ROOT = Path("/content/drive/MyDrive") / PERSIST_FOLDER
    RUNTIME = "Google Colab"
elif IN_KAGGLE:
    WORK_ROOT = Path("/kaggle/working")
    PERSIST_ROOT = WORK_ROOT / PERSIST_FOLDER
    RUNTIME = "Kaggle"
else:
    WORK_ROOT = Path.cwd()
    PERSIST_ROOT = WORK_ROOT / PERSIST_FOLDER
    RUNTIME = "Other/Jupyter"

PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
print("runtime:", RUNTIME)
print("work root:", WORK_ROOT)
print("persistent root:", PERSIST_ROOT)


## 3. Verify the GPU

If this cell says CUDA is unavailable, enable a GPU accelerator before continuing.


In [ ]:
import subprocess
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Enable GPU in Colab (Runtime > Change runtime type) "
        "or Kaggle (Notebook options > Accelerator > GPU), then restart from this cell."
    )

print("GPU:", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
subprocess.run(["nvidia-smi"], check=False)


## 4. Clone/update RehabLLM and install cloud dependencies


In [ ]:
import shutil
import subprocess

REPO_DIR = WORK_ROOT / "RehabLLM"

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[cloud]"],
    check=True,
)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repo:", REPO_DIR)
print("commit:", commit)


## 5. Restore or build the rehabilitation corpus

The notebook first looks for a prepared persistent data cache. If none exists, it streams the filtered/deduplicated Common Pile PubMed corpus, selects rehabilitation/robotics material, applies conservative licence/quality checks, trains the tokenizer, and creates train/validation/test token files.


In [ ]:
DATA_CACHE = PERSIST_ROOT / "data_cache"
LOCAL_CURATED = REPO_DIR / "data" / "curated"
LOCAL_PROCESSED = REPO_DIR / "data" / "processed"

required_cache = [
    DATA_CACHE / "processed" / "train.bin",
    DATA_CACHE / "processed" / "val.bin",
    DATA_CACHE / "processed" / "test.bin",
    DATA_CACHE / "processed" / "rehab_sp.model",
    DATA_CACHE / "processed" / "rehab_sp.vocab",
    DATA_CACHE / "curated" / "gpu_corpus_stats.json",
]

cache_ready = all(path.exists() for path in required_cache) and not FORCE_REBUILD_DATA
if cache_ready:
    print("Restoring prepared data cache...")
    shutil.copytree(DATA_CACHE / "curated", LOCAL_CURATED, dirs_exist_ok=True)
    shutil.copytree(DATA_CACHE / "processed", LOCAL_PROCESSED, dirs_exist_ok=True)
else:
    print("Building rehabilitation corpus from streaming PubMed data...")
    subprocess.run([
        sys.executable, "scripts/build_hf_rehab_corpus.py",
        "--target-docs", str(TARGET_DOCS),
        "--out-dir", "data/curated",
    ], check=True)

    subprocess.run([
        sys.executable, "scripts/train_tokenizer.py",
        "--input", "data/curated/train.txt",
        "--prefix", "data/processed/rehab_sp",
        "--vocab-size", "8000",
    ], check=True)

    subprocess.run([
        sys.executable, "scripts/prepare_data.py",
        "--train-text", "data/curated/train.txt",
        "--val-text", "data/curated/val.txt",
        "--test-text", "data/curated/test.txt",
        "--tokenizer", "data/processed/rehab_sp.model",
    ], check=True)

    if DATA_CACHE.exists():
        shutil.rmtree(DATA_CACHE)
    (DATA_CACHE / "curated").mkdir(parents=True, exist_ok=True)
    (DATA_CACHE / "processed").mkdir(parents=True, exist_ok=True)
    for name in ("train.txt", "val.txt", "test.txt", "gpu_corpus_stats.json"):
        source = LOCAL_CURATED / name
        if source.exists():
            shutil.copy2(source, DATA_CACHE / "curated" / name)
    for name in ("train.bin", "val.bin", "test.bin", "rehab_sp.model", "rehab_sp.vocab"):
        shutil.copy2(LOCAL_PROCESSED / name, DATA_CACHE / "processed" / name)
    print("Prepared data cached at:", DATA_CACHE)


## 6. Inspect corpus and token counts


In [ ]:
import json
import numpy as np

stats_path = LOCAL_CURATED / "gpu_corpus_stats.json"
if stats_path.exists():
    stats = json.loads(stats_path.read_text())
    print(json.dumps(stats, indent=2))

for split in ("train", "val", "test"):
    path = LOCAL_PROCESSED / f"{split}.bin"
    tokens = path.stat().st_size // np.dtype(np.int32).itemsize
    print(f"{split:>5}: {tokens:,} tokens")


## 7. Train — automatically resume from the newest checkpoint

This is the long-running cell. Intermediate checkpoints are written every **500 steps**. If the free runtime disconnects, reopen the notebook and run the cells again; `--resume latest` continues from the newest saved checkpoint when persistent storage is available.


In [ ]:
CHECKPOINT_DIR = PERSIST_ROOT / "checkpoints" / "free_gpu_v0.3"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_CHECKPOINT = CHECKPOINT_DIR / "final.pt"

if FINAL_CHECKPOINT.exists():
    print("Training already complete:", FINAL_CHECKPOINT)
else:
    subprocess.run([
        sys.executable, "scripts/train_model.py",
        "--config", CONFIG_PATH,
        "--train", "data/processed/train.bin",
        "--val", "data/processed/val.bin",
        "--out", str(CHECKPOINT_DIR),
        "--resume", "latest",
    ], check=True)

print("checkpoint directory:", CHECKPOINT_DIR)
print("final exists:", FINAL_CHECKPOINT.exists())


## 8. Evaluate the completed model


In [ ]:
EVAL_DIR = PERSIST_ROOT / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_PATH = EVAL_DIR / "free_gpu_v0.3.json"

if not FINAL_CHECKPOINT.exists():
    print("No final checkpoint yet. Resume training before evaluation.")
else:
    subprocess.run([
        sys.executable, "scripts/evaluate_model.py",
        "--checkpoint", str(FINAL_CHECKPOINT),
        "--tokenizer", "data/processed/rehab_sp.model",
        "--test", "data/processed/test.bin",
        "--out", str(EVAL_PATH),
        "--batch-size", "16",
        "--eval-batches", "50",
    ], check=True)
    result = json.loads(EVAL_PATH.read_text())
    print("test loss:", result["test_loss"])
    print("test perplexity:", result["test_perplexity"])


## 9. Try the trained RehabLLM


In [ ]:
if not FINAL_CHECKPOINT.exists():
    print("Complete training first.")
else:
    subprocess.run([
        sys.executable, "scripts/generate_text.py",
        "--checkpoint", str(FINAL_CHECKPOINT),
        "--tokenizer", "data/processed/rehab_sp.model",
        "--prompt", EVAL_PROMPT,
        "--max-new-tokens", "160",
    ], check=True)


## 10. Package the important artefacts

The ZIP contains the final checkpoint (when available), run manifest, tokenizer, evaluation output, corpus statistics and training configuration. Intermediate checkpoints remain in the persistent checkpoint directory for resume/recovery.


In [ ]:
PACKAGE_DIR = PERSIST_ROOT / "package"
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True)

package_sources = [
    FINAL_CHECKPOINT,
    CHECKPOINT_DIR / "run_manifest.json",
    LOCAL_PROCESSED / "rehab_sp.model",
    LOCAL_PROCESSED / "rehab_sp.vocab",
    LOCAL_CURATED / "gpu_corpus_stats.json",
    EVAL_PATH,
    REPO_DIR / CONFIG_PATH,
]
for source in package_sources:
    if Path(source).exists():
        shutil.copy2(source, PACKAGE_DIR / Path(source).name)

archive_base = PERSIST_ROOT / "RehabLLM_free_gpu_v0.3"
archive = shutil.make_archive(str(archive_base), "zip", root_dir=PACKAGE_DIR)
print("archive:", archive)
print("persistent checkpoint directory:", CHECKPOINT_DIR)


## Resume notes

- **Colab:** reopen the notebook, mount the same Google Drive, and run all cells. The prepared data cache and latest checkpoint are restored automatically.
- **Kaggle:** save a notebook version so `/kaggle/working` becomes output. For a later session, restore/attach the previous `RehabLLM-FreeGPU` output before the training cell if Kaggle does not restore it automatically.
- Training is complete only when `final.pt` exists.
